# Task 1 · Traders having an unusually bad session

**Question:** who is having a bad session today *relative to what is normal for that trader*?

Plan:
1. Load the data and compute P&L per trader and session (`load.py`, `pnl.py`).
2. Merge everything into one table per trader and session (broker, account, limits).
3. Exclude traders with fewer than 5 sessions of history: without history there is no "normal" to compare against.
4. Descriptive statistics of each trader's history (`describe()`).
5. Normalise size with the point value and compare today against each trader's normal.
6. Score, rank and pick ten; check how sensitive the list is to the thresholds; backtest it.

## What "unusual behaviour" can look like

| Signal | In plain words | How it is measured |
| --- | --- | --- |
| **Loss outside their range** | Lost much more than they usually lose | Today's P&L vs. their own history (z-score / worst day) |
| **Bigger bet than usual** | Increased the size of the position | Exposure in USD (contracts × point value × price) vs. their normal |
| **Product switch** | Moved from a micro to a full-size contract (10× bigger) or to another market | Today's instruments vs. the ones they always use |
| **Overtrading** | Many more trades than usual | Orders per session vs. their normal |

## 1. Load and per-session P&L

In [1]:
import numpy as np
import pandas as pd

from load import CUT, load
from pnl import account_sessions, trader_sessions

pd.set_option("display.width", 220, "display.max_columns", 40, "display.float_format", "{:,.2f}".format)

d = load()
fills, accounts, traders, brokers = d["fills"], d["accounts"], d["traders"], d["brokers"]
acct = account_sessions(fills, d["marks"])
TODAY = fills["session"].max()
print("cut:", CUT, "| today's session:", TODAY.date(), "| fills:", len(fills))

cut: 2026-08-25 14:30:00+00:00 | today's session: 2026-08-25 | fills: 5146


## 2. Position size, normalised with the point value

Counting contracts is misleading: 3 CL contracts are not the same as 3 MCL. Two measures that are comparable:

- **`max_usd_per_point`** = open contracts × point value. *How many dollars they make or lose if the market moves 1 point.* Comparable within one market (MCL vs CL, MES vs ES).
- **`max_exposure_usd`** = open contracts × point value × price. *The total size of the position in dollars.* Comparable across markets (crude vs S&P).

We use the session **maximum**: the moment of highest risk.

In [2]:
acct["max_usd_per_point"] = acct["max_position"] * acct["point_value_usd"]

# Max per instrument, then max per trader and session.
# (Simplification: the peak of each instrument, not the simultaneous peak across instruments.)
size = acct.groupby(["trader_id", "session"]).agg(
    max_usd_per_point=("max_usd_per_point", "max"),
    max_contracts=("max_position", "max"),
).reset_index()
size.head()

,trader_id,session,max_usd_per_point,max_contracts
0,T-001,2026-08-10,200,10
1,T-001,2026-08-11,200,10
2,T-001,2026-08-12,200,10
3,T-001,2026-08-13,400,8
4,T-001,2026-08-14,500,10


## 3. One table per trader and session (merge)

In [3]:
ts = trader_sessions(acct).merge(size, on=["trader_id", "session"], how="left")

# Account data, aggregated per trader (5 traders have more than one account)
acct_info = accounts.groupby("trader_id").agg(
    n_accounts=("id", "nunique"),
    total_balance=("balance", "sum"),
    max_position_size=("max_position_size", "max"),
    statuses=("status", lambda s: ",".join(sorted(set(s)))),
).reset_index()

table = (
    ts.merge(acct_info, on="trader_id", how="left", validate="many_to_one")
      .merge(traders[["id", "broker_id"]].rename(columns={"id": "trader_id"}), on="trader_id", how="left")
      .merge(brokers[["id", "name"]].rename(columns={"id": "broker_id", "name": "broker"}), on="broker_id", how="left")
      .rename(columns={
          "net_pnl_usd": "net_pnl", "realized_usd": "realized_pnl", "unrealized_usd": "open_pnl",
          "commission_usd": "commissions", "n_orders": "orders", "n_fills": "fills",
          "contracts": "contracts_traded", "max_notional_usd": "max_exposure_usd",
      })
)
print(table.shape)
table.head()

(516, 25)


,trader_id,session,net_pnl,realized_pnl,open_pnl,commissions,fills,orders,contracts_traded,max_exposure_usd,open_positions,instruments,account_types,accounts,first_fill,last_fill,is_today,max_usd_per_point,max_contracts,n_accounts,total_balance,max_position_size,statuses,broker_id,broker
0,T-001,2026-08-10,-628.20,-435.00,0.00,193.20,12,12,92,"3,945,450.00",0,NQ,eval,1,2026-08-10 10:35:25+00:00,2026-08-10 20:15:01+00:00,False,200,10,1,"100,000.00",10,active,BRK-ARWP,ArrowFin Retail (ECX)
1,T-001,2026-08-11,-664.80,-480.00,0.00,184.80,12,10,88,"3,947,650.00",0,NQ,eval,1,2026-08-11 10:34:24+00:00,2026-08-11 20:02:23+00:00,False,200,10,1,"100,000.00",10,active,BRK-ARWP,ArrowFin Retail (ECX)
2,T-001,2026-08-12,"-2,226.00","-1,995.00",0.00,231.00,14,14,110,"3,949,800.00",0,NQ,eval,1,2026-08-12 10:38:47+00:00,2026-08-12 20:06:25+00:00,False,200,10,1,"100,000.00",10,active,BRK-ARWP,ArrowFin Retail (ECX)
3,T-001,2026-08-13,"2,319.80","2,450.00",0.00,130.20,11,10,62,"2,247,300.00",0,ES,eval,1,2026-08-13 10:43:49+00:00,2026-08-13 20:16:20+00:00,False,400,8,1,"100,000.00",10,active,BRK-ARWP,ArrowFin Retail (ECX)
4,T-001,2026-08-14,"-3,010.10","-2,787.50",0.00,222.60,15,14,106,"2,810,000.00",0,ES,eval,1,2026-08-14 10:39:01+00:00,2026-08-14 20:19:11+00:00,False,500,10,1,"100,000.00",10,active,BRK-ARWP,ArrowFin Retail (ECX)


## 4. Who has enough history?

Rule: **traders with fewer than 5 sessions before today are excluded.** With 1 or 2 days we do not know what "normal" is for that person.
We also list the demo account (play money) and the closed/restricted accounts.

In [4]:
n_hist = table[table["session"] < TODAY].groupby("trader_id")["session"].nunique()
table["history_sessions"] = table["trader_id"].map(n_hist).fillna(0).astype(int)

excluded = table[table["history_sessions"] < 5].groupby("trader_id").agg(
    history_sessions=("history_sessions", "first"), account_types=("account_types", "first"),
    pnl_today=("net_pnl", lambda s: s[table.loc[s.index, "session"] == TODAY].sum()),
)
print("Excluded for short history:")
print(excluded)

print("\nSpecial accounts:")
print(accounts[(accounts.account_type == "demo") | (accounts.status != "active")]
      [["id", "trader_id", "account_type", "status", "balance", "buying_power"]].to_string(index=False))

table = table[table["history_sessions"] >= 5].copy()
hist = table[table["session"] < TODAY]
today = table[table["session"] == TODAY].set_index("trader_id")
print(f"\n{table.trader_id.nunique()} traders remain; {len(hist)} history sessions; {len(today)} active today")

Excluded for short history:
           history_sessions account_types  pnl_today
trader_id                                           
T-024                     0        funded  -1,579.60
T-025                     0        funded     -72.00
T-026                     1          eval    -127.95

Special accounts:
      id trader_id account_type     status    balance  buying_power
ACC-2047     T-043         demo     active 100,000.00    100,000.00
ACC-2053     T-040         eval     closed  31,500.00          0.00
ACC-2054     T-041       funded restricted  25,000.00          0.00

45 traders remain; 467 history sessions; 45 active today


The demo account (T-043) and the restricted account (ACC-2054, T-041, which still trades today with zero buying power) stay in the scoring: the rules apply the same way, and neither ends up in the top ten. The closed account (ACC-2053, T-040) has no activity today.

## 5. Each trader's history: descriptive statistics

First the overall picture (all historical sessions together), then per trader.

In [5]:
metrics = ["net_pnl", "orders", "contracts_traded", "max_usd_per_point", "max_exposure_usd", "commissions"]
hist[metrics].describe()

,net_pnl,orders,contracts_traded,max_usd_per_point,max_exposure_usd,commissions
count,467.00,467.00,467.00,467.00,467.00,467.00
mean,-76.76,9.45,36.15,"48,035.27","591,184.47",42.94
std,865.24,7.03,32.95,"136,788.32","982,366.62",46.99
min,"-5,053.95",4.00,4.00,2.00,"7,397.00",2.80
25%,-260.45,6.00,14.00,10.00,"112,410.00",12.60
50%,-35.50,8.00,22.00,30.00,"197,286.25",23.80
75%,193.70,10.00,58.00,400.00,"549,250.00",51.80
max,"4,227.80",48.00,180.00,"500,000.00","3,967,050.00",231.00


**What the overall picture says**

- The typical session is close to flat: median **−$35**, and half of all sessions fall between −$260 and +$194. But the standard deviation is **$865**, with days of −$5,054 and +$4,228: some traders operate on a very different scale.
- Max exposure ranges from **$7 thousand to $4 million**, a factor of more than 500 between the smallest and the largest trader. **A single dollar threshold for everyone** (like "lost more than $1,000") cannot work.
- `max_usd_per_point` reaches $500,000 because of 6E (euro), whose "point" is worth $125,000. This measure is only comparable within one market; **across markets we use `max_exposure_usd`**.
- Half of all sessions have between 6 and 10 orders; the maximum is 48 (one high-frequency trader).

In [6]:
# Net P&L per trader: describe() of their history (excluding today)
pnl_desc = hist.groupby("trader_id")["net_pnl"].describe()
pnl_desc.sort_values("std", ascending=False)

,count,mean,std,min,25%,50%,75%,max
trader_id,,,,,,,,
T-004,11.00,-363.33,"2,474.42","-2,890.30","-2,655.45",-117.70,"1,231.60","4,227.80"
T-007,11.00,64.39,"2,108.41","-4,536.20",-777.35,669.80,"1,508.90","2,647.70"
T-006,11.00,-536.90,"2,076.31","-3,966.00","-2,512.80",299.70,"1,168.80","1,642.80"
T-001,11.00,-169.68,"1,903.53","-3,010.10","-1,445.40",-294.40,"1,483.97","2,319.80"
T-002,11.00,"-1,351.76","1,895.68","-5,053.95","-1,856.15",-897.00,-545.05,"1,564.80"
T-003,11.00,-403.88,"1,666.34","-2,662.15","-1,746.35",-464.10,715.65,"2,913.60"
T-005,11.00,116.39,"1,644.14","-1,639.00",-926.35,-414.30,697.80,"3,189.60"
T-017,11.00,-118.22,595.83,"-1,278.00",-441.40,-275.50,341.43,830.50
T-041,10.00,-47.60,594.45,-836.80,-474.05,-128.85,351.42,"1,068.60"


**The key column is `std`** (how much their P&L normally moves from one day to the next):

- T-004, T-007, T-006, T-001, T-002 have a `std` of **~$2,000**: for them, losing $2,000 in a day is routine.
- At the bottom of the table, traders have a `std` of **$100–$200**: for them, losing $1,500 never happens.

This is exactly what the brief asks for: the same dollar loss means very different things for different traders. So each trader is compared against their own `std`.

Caveat: each trader has only 8–11 sessions of history. A `std` from so few points is a noisy estimate, so we complement it with "previous worst day", which is easier to interpret.

In [7]:
# Orders per session, per trader
hist.groupby("trader_id")["orders"].describe().sort_values("mean", ascending=False)

,count,mean,std,min,25%,50%,75%,max
trader_id,,,,,,,,
T-013,11.00,48.00,0.00,48.00,48.00,48.00,48.00,48.00
T-016,11.00,18.00,0.00,18.00,18.00,18.00,18.00,18.00
T-015,11.00,18.00,0.00,18.00,18.00,18.00,18.00,18.00
T-014,11.00,18.00,0.00,18.00,18.00,18.00,18.00,18.00
T-018,8.00,12.00,0.00,12.00,12.00,12.00,12.00,12.00
T-001,11.00,11.45,2.21,8.00,10.00,12.00,13.00,14.00
T-006,11.00,11.45,2.38,8.00,10.00,12.00,14.00,14.00
T-007,11.00,11.27,2.41,8.00,9.00,12.00,13.00,14.00
T-005,11.00,11.09,2.07,8.00,10.00,12.00,12.00,14.00


Most traders place **a very stable number of orders** from one day to the next (low `std`; several place exactly the same number every day, `std` = 0).
That makes a change in order count a clean signal: if someone who always places 6 places 10 today, it is unusual *for them*.

In [8]:
# Normalised size: max exposure in USD per trader
hist.groupby("trader_id")["max_exposure_usd"].describe().sort_values("50%", ascending=False)

,count,mean,std,min,25%,50%,75%,max
trader_id,,,,,,,,
T-022,11.00,"3,348,820.45","87,051.92","3,086,875.00","3,370,050.00","3,374,250.00","3,378,525.00","3,383,850.00"
T-001,11.00,"2,707,114.77","1,384,221.06","281,200.00","2,249,700.00","2,810,000.00","3,946,550.00","3,962,600.00"
T-004,11.00,"2,076,487.91","1,452,928.55","280,750.00","395,520.00","2,804,625.00","2,815,625.00","3,964,800.00"
T-007,11.00,"1,878,741.18","1,544,552.58","280,925.00","375,505.25","2,529,562.50","2,986,580.00","3,964,700.00"
T-037,11.00,"794,091.59","284,613.72","274,887.50","549,912.50","989,800.00","993,360.00","994,800.00"
T-045,9.00,"704,296.11","256,497.58","275,750.00","549,725.00","743,940.00","990,400.00","994,920.00"
T-041,10.00,"568,611.80","348,359.01","158,072.00","180,751.00","743,445.00","745,507.50","994,720.00"
T-033,9.00,"652,120.14","271,359.37","275,512.50","549,700.00","550,550.00","990,320.00","994,880.00"
T-028,11.00,"701,271.25","290,286.36","275,637.50","480,987.50","550,550.00","992,300.00","994,840.00"


Two kinds of trader by size:

- **Stable**: always the same size (T-014, T-015 with a `std` of ~$360 on ~$198 thousand; T-018, T-023). Any jump is very visible.
- **Bimodal**: they alternate between small and large contracts (T-001 to T-007: min ~$280 thousand, max ~$4 million). For them the median is misleading: a "big" day looks 9× normal even though they have done it before.

So for risk we use two comparisons: against the **median** (bigger than a typical day?) and against the **historical max** (bigger than *any* previous day?). We only call it unusual if it exceeds their max.

## 6. Caveat: today's session is not over

The cut is at 9:30 Chicago time and the session closes at 16:00. Comparing today's order count against full days would **understate** today's activity.
What fraction of a normal day's activity has usually happened by this time?

In [9]:
CT = "America/Chicago"
session_open = (fills["session"] - pd.Timedelta(days=1)).dt.tz_localize(CT) + pd.Timedelta(hours=17)
fills["minutes_since_open"] = (fills["filled_at"].dt.tz_convert(CT) - session_open).dt.total_seconds() / 60
cut_minutes = (CUT.tz_convert(CT) - ((TODAY - pd.Timedelta(days=1)).tz_localize(CT) + pd.Timedelta(hours=17))).total_seconds() / 60

past = fills[fills["session"] < TODAY]
frac = (past["minutes_since_open"] <= cut_minutes).mean()
print(f"Today is {cut_minutes/60:.1f} h into a 23 h session. On previous days, {frac:.0%} of fills had happened by this time.")

Today is 16.5 h into a 23 h session. On previous days, 42% of fills had happened by this time.


By that measure, less than half of a normal day should have happened by now. Yet today already has **more fills than an average full day**. Let us look at when they happened:

In [10]:
fills["hour_chicago"] = fills["filled_at"].dt.tz_convert(CT).dt.hour
by_hour = pd.crosstab(fills["hour_chicago"], np.where(fills["session"] == TODAY, "today", "history"), normalize="columns")
print(by_hour.round(3))
print("\nfills today:", (fills["session"] == TODAY).sum(),
      "| average full previous day:", round(past.groupby("session").size().mean()))
print("last fill today (Chicago):", fills.loc[fills["session"] == TODAY, "filled_at"].max().tz_convert(CT))

col_0         history  today
hour_chicago                
1                0.00   0.05
2                0.00   0.17
3                0.00   0.09
4                0.00   0.09
5                0.19   0.12
6                0.06   0.11
7                0.06   0.07
8                0.08   0.25
9                0.05   0.02
10               0.13   0.00
11               0.07   0.00
12               0.08   0.00
13               0.04   0.00
14               0.16   0.00
15               0.08   0.00
17               0.00   0.01
18               0.00   0.01

fills today: 497 | average full previous day: 423
last fill today (Chicago): 2026-08-25 09:19:39-05:00


**Data finding:** today's session does not have the shape of a normal day.

- In the history there are **never** any fills between 1:00 and 4:59 a.m. Chicago; today ~40% of fills fall there.
- Nothing happens after 9:20 today, yet there are already more fills than an average full day.

Today's session looks like a full day of activity "squeezed" into the morning (probably an artefact of how the synthetic data was generated). Therefore:

1. **We compare today against full days**, not against "the same time of day". We tried the same-time comparison and it made *every* trader look 2–3× more active than normal: when everyone looks unusual, the problem is the comparison, not the traders.
2. **We do not use time of day as a signal**: today everyone trades at unusual hours.
3. Even so, today's P&L is a **snapshot at 9:30**; on a real day it could still change.

This would be reported to the data team before trusting any time-based signal.

## 7. Today vs. their normal

For each trader we compare today against **their own** history:

- **`pnl_z`**: how many of *their own* standard deviations today's P&L is below their mean. −3 = very unusual for that person.
- **`pnl_vs_worst_day`**: today's P&L divided by their previous worst day. >1 = today is already their worst day on record.
- **`orders_vs_normal`**, **`contracts_vs_normal`**, **`risk_vs_normal`**: today / median of their history (full days). 1.0 = same as always.
- **`risk_vs_max`**: today's exposure / their maximum historical exposure. >1 = they have never risked this much.
- **`new_product`**: instruments they traded today and never before.
- **`evidence`**: one sentence describing what we see, so a human understands it without reading the table.

`risk_vs_normal` uses exposure in USD, which already includes the point value. If `contracts_vs_normal ≈ 1` but `risk_vs_normal` is high, the trader **switched to a bigger contract** even though the count looks the same.

In [11]:
g = hist.groupby("trader_id")
base = pd.DataFrame({
    "pnl_mean": g["net_pnl"].mean(),
    "pnl_std": g["net_pnl"].std(),
    "pnl_worst": g["net_pnl"].min(),
    "orders_median": g["orders"].median(),
    "contracts_median": g["contracts_traded"].median(),
    "exposure_median": g["max_exposure_usd"].median(),
    "exposure_max_hist": g["max_exposure_usd"].max(),
    "usual_instruments": g["instruments"].agg(lambda s: set(",".join(s).split(","))),
})

cmp = today.join(base, how="inner")
cmp["pnl_z"] = (cmp["net_pnl"] - cmp["pnl_mean"]) / cmp["pnl_std"]
cmp["pnl_vs_worst_day"] = np.where(cmp["pnl_worst"] < 0, cmp["net_pnl"] / cmp["pnl_worst"], np.nan)
cmp["orders_vs_normal"] = cmp["orders"] / cmp["orders_median"]
cmp["contracts_vs_normal"] = cmp["contracts_traded"] / cmp["contracts_median"]
cmp["risk_vs_normal"] = cmp["max_exposure_usd"] / cmp["exposure_median"]
cmp["risk_vs_max"] = cmp["max_exposure_usd"] / cmp["exposure_max_hist"]
cmp["new_product"] = [
    ",".join(sorted(set(i.split(",")) - h)) for i, h in zip(cmp["instruments"], cmp["usual_instruments"])
]


def evidence(r):
    out = []
    if r.pnl_z <= -2:
        out.append(f"lost ${-r.net_pnl:,.0f}; normally moves ±${r.pnl_std:,.0f} (z={r.pnl_z:.1f})")
    if r.pnl_vs_worst_day > 1:
        out.append(f"already their worst session ({r.pnl_vs_worst_day:.1f}x their previous worst day)")
    if r.new_product:
        out.append(f"traded a product they had never used: {r.new_product}")
    if r.risk_vs_max > 1.2:
        if r.contracts_vs_normal < 1.5:
            out.append(f"same contract count ({r.contracts_vs_normal:.1f}x) but {r.risk_vs_normal:.1f}x more money at risk than normal, more than ever before")
        else:
            out.append(f"{r.risk_vs_normal:.1f}x more money at risk than normal, more than ever before")
    if r.orders_vs_normal >= 1.5:
        out.append(f"{r.orders:.0f} orders vs. the {r.orders_median:.0f} they usually place in a day")
    if r.open_pnl < 0:
        out.append(f"holds an open position losing ${-r.open_pnl:,.0f}")
    return "; ".join(out)


cmp["evidence"] = cmp.apply(evidence, axis=1)
cols = ["net_pnl", "pnl_std", "pnl_z", "pnl_vs_worst_day", "orders", "orders_vs_normal",
        "contracts_vs_normal", "risk_vs_normal", "risk_vs_max", "new_product", "open_pnl", "evidence"]
with pd.option_context("display.max_colwidth", 200):
    display(cmp.sort_values("pnl_z")[cols].head(15))

,net_pnl,pnl_std,pnl_z,pnl_vs_worst_day,orders,orders_vs_normal,contracts_vs_normal,risk_vs_normal,risk_vs_max,new_product,open_pnl,evidence
trader_id,,,,,,,,,,,,
T-018,"-1,516.80",157.81,-8.91,4.32,12,1.00,0.86,0.67,0.67,,0.00,"lost $1,517; normally moves ±$158 (z=-8.9); already their worst session (4.3x their previous worst day)"
T-023,"-2,486.10",387.07,-5.45,2.70,5,0.83,0.95,7.42,5.95,CL,"-2,340.00","lost $2,486; normally moves ±$387 (z=-5.4); already their worst session (2.7x their previous worst day); traded a product they had never used: CL; same contract count (0.9x) but 7.4x more money at..."
T-019,"-1,559.20",305.19,-5.08,3.15,8,1.00,1.08,1.00,1.00,,0.00,"lost $1,559; normally moves ±$305 (z=-5.1); already their worst session (3.2x their previous worst day)"
T-009,-669.95,121.31,-4.97,2.66,6,1.00,1.20,3.34,2.38,ES,0.00,lost $670; normally moves ±$121 (z=-5.0); already their worst session (2.7x their previous worst day); traded a product they had never used: ES; same contract count (1.2x) but 3.3x more money at r...
T-011,-758.55,165.86,-4.62,3.08,8,1.33,1.00,1.00,0.71,,0.00,lost $759; normally moves ±$166 (z=-4.6); already their worst session (3.1x their previous worst day)
T-012,-724.10,174.18,-4.28,2.26,8,1.00,1.29,1.01,1.00,,0.00,lost $724; normally moves ±$174 (z=-4.3); already their worst session (2.3x their previous worst day)
T-020,"-2,397.65",541.69,-4.24,1.89,10,1.67,1.86,1.01,1.00,,0.00,"lost $2,398; normally moves ±$542 (z=-4.2); already their worst session (1.9x their previous worst day); 10 orders vs. the 6 they usually place in a day"
T-010,-663.85,179.58,-3.36,1.85,8,1.33,1.80,0.71,0.71,,0.00,lost $664; normally moves ±$180 (z=-3.4); already their worst session (1.8x their previous worst day)
T-008,-459.00,151.29,-2.84,2.20,4,0.67,0.83,1.41,1.00,,0.00,lost $459; normally moves ±$151 (z=-2.8); already their worst session (2.2x their previous worst day)


### The T-023 case: `risk_vs_normal` shows it

In [12]:
t23 = table[table.trader_id == "T-023"][
    ["session", "instruments", "contracts_traded", "max_usd_per_point", "max_exposure_usd", "net_pnl", "open_pnl"]
]
t23

,session,instruments,contracts_traded,max_usd_per_point,max_exposure_usd,net_pnl,open_pnl
259,2026-08-10,MCL,20,500,"36,850.00",-858.00,0.00
260,2026-08-11,MCL,20,400,"29,524.00",-84.00,0.00
261,2026-08-12,MCL,22,500,"36,895.00",-541.40,0.00
262,2026-08-13,MCL,24,500,"36,935.00",-312.80,0.00
263,2026-08-14,MCL,18,500,"36,950.00",-685.60,0.00
264,2026-08-17,MCL,20,400,"29,588.00",-634.00,0.00
265,2026-08-18,MCL,20,400,"29,604.00",-62.00,0.00
266,2026-08-19,MCL,20,400,"29,604.00",-344.00,0.00
267,2026-08-20,MCL,18,400,"29,636.00",-23.60,0.00
268,2026-08-21,MCL,22,500,"37,030.00",-919.40,0.00


In [13]:
for c in ["contracts_vs_normal", "risk_vs_normal", "risk_vs_max", "new_product", "evidence"]:
    print(f"{c:>20}: {cmp.loc['T-023', c]}")

 contracts_vs_normal: 0.95
      risk_vs_normal: 7.422138330861535
         risk_vs_max: 5.9465298406697285
         new_product: CL
            evidence: lost $2,486; normally moves ±$387 (z=-5.4); already their worst session (2.7x their previous worst day); traded a product they had never used: CL; same contract count (0.9x) but 7.4x more money at risk than normal, more than ever before; holds an open position losing $2,340


Counting contracts, T-023 traded **as always** (0.95x: 19 contracts vs. ~20). But they moved from MCL (point value $100) to CL ($1,000): max exposure was **$220 thousand vs. ~$30 thousand normally (7.4x)**, almost 6× their historical max. Without normalising by the point value, this case goes unnoticed.

## 8. Risk relative to the account

So far each trader was compared with themselves in dollars. Now we compare against **the size of their account and its rules**, which is what matters to the firm (break a rule and the account is liquidated).

Three ratios, per account and session:

| Measure | In plain words | Formula |
| --- | --- | --- |
| **`limit_usage`** | How close they came to the max contracts their account allows. 1.0 = at the limit; >1 = over it | max open contracts / `max_position_size` |
| **`leverage`** | How many times the account size they have at risk. 10 = moving 10× the money they have | max exposure USD / `balance` |
| **`pnl_pct_account`** | Share of the account won or lost in the session | net P&L / `balance` |

Computed **per account** (not per trader), because rules belong to accounts. For each one we compare **today** with its historical **median** and **max**.

Limitations:
- We do not know whether `max_position_size` counts a micro the same as a full-size contract (some firms count 10 micros = 1 contract). Here each contract counts as 1.
- "Max open contracts" is the per-instrument peak, not the simultaneous sum across instruments.
- We do not have each account's daily loss limit or drawdown floor, so `pnl_pct_account` approximates "how close to the rule".

In [14]:
risk = acct.groupby(["trader_id", "account_id", "session"]).agg(
    max_contracts=("max_position", "max"),
    max_exposure_usd=("max_notional_usd", "max"),
    net_pnl=("net_pnl_usd", "sum"),
).reset_index().merge(
    accounts[["id", "account_type", "status", "balance", "max_position_size"]].rename(columns={"id": "account_id"}),
    on="account_id", how="left",
)
print("Accounts with a zero balance (ratio cannot be computed):",
      accounts.loc[accounts.balance <= 0, ["id", "trader_id"]].values.tolist())
risk = risk[risk["balance"] > 0]
risk["limit_usage"] = risk["max_contracts"] / risk["max_position_size"]
risk["leverage"] = risk["max_exposure_usd"] / risk["balance"]
risk["pnl_pct_account"] = risk["net_pnl"] / risk["balance"]

risk_hist = risk[risk["session"] < TODAY].groupby("account_id")
risk_today = risk[risk["session"] == TODAY].set_index("account_id")
prop = pd.DataFrame({
    "trader_id": risk_today["trader_id"],
    "type": risk_today["account_type"],
    "status": risk_today["status"],
    "balance": risk_today["balance"],
    "contract_limit": risk_today["max_position_size"],
    "limit_usage_median": risk_hist["limit_usage"].median(),
    "limit_usage_max": risk_hist["limit_usage"].max(),
    "limit_usage_today": risk_today["limit_usage"],
    "leverage_median": risk_hist["leverage"].median(),
    "leverage_max": risk_hist["leverage"].max(),
    "leverage_today": risk_today["leverage"],
    "pnl_pct_worst": risk_hist["pnl_pct_account"].min(),
    "pnl_pct_today": risk_today["pnl_pct_account"],
    "history_sessions": risk_hist.size(),
})
prop = prop[prop["trader_id"].isin(table["trader_id"])]  # same traders as above (>= 5 sessions)
prop["leverage_today_vs_normal"] = prop["leverage_today"] / prop["leverage_median"]

Accounts with a zero balance (ratio cannot be computed): [['ACC-2028', 'T-025']]


### 8.1 Who exceeded their contract limit?

In [15]:
over = risk[(risk["limit_usage"] > 1) & risk["trader_id"].isin(table["trader_id"])]
over.groupby(["trader_id", "account_id"]).agg(
    limit=("max_position_size", "first"),
    sessions_over_limit=("session", "size"),
    total_sessions=("account_id", lambda s: (risk["account_id"] == s.iloc[0]).sum()),
    worst_usage=("limit_usage", "max"),
    over_limit_today=("session", lambda s: (s == TODAY).any()),
)

,,limit,sessions_over_limit,total_sessions,worst_usage,over_limit_today
trader_id,account_id,,,,,
T-013,ACC-2013,5,7,12,1.80,True
T-022,ACC-2025,20,1,12,1.60,True


- **T-013** exceeded their 5-contract limit in 7 of 12 sessions (up to 1.8x). Not unusual *for them*, but a **recurring rule breach**: a controls problem, not a "bad day". It should not drop off the radar just because "they always do it".
- **T-022** had never gone above 60% of their limit and **today went over it (1.6x: 32 contracts against a limit of 20)**. Their P&L today is ~$0, so a loss-based list would not see them. This is exactly the "trouble coming" the product wants to catch before money is lost.

### 8.2 Leverage and loss as % of the account: today vs. history

In [16]:
cols_prop = ["trader_id", "type", "balance", "contract_limit", "limit_usage_max", "limit_usage_today",
             "leverage_median", "leverage_max", "leverage_today", "leverage_today_vs_normal", "pnl_pct_worst", "pnl_pct_today"]
fmt = {c: "{:.0%}" for c in ["limit_usage_max", "limit_usage_today"]}
fmt.update({c: "{:.1%}" for c in ["pnl_pct_worst", "pnl_pct_today"]})
fmt.update({c: "{:.1f}x" for c in ["leverage_median", "leverage_max", "leverage_today", "leverage_today_vs_normal"]})
fmt.update({"balance": "${:,.0f}", "contract_limit": "{:.0f}"})
prop.sort_values("pnl_pct_today")[cols_prop].head(20).style.format(fmt)

,trader_id,type,balance,contract_limit,limit_usage_max,limit_usage_today,leverage_median,leverage_max,leverage_today,leverage_today_vs_normal,pnl_pct_worst,pnl_pct_today
account_id,,,,,,,,,,,,
ACC-2017,T-017,funded,"$18,000",8,88%,88%,10.9x,11.0x,11.0x,1.0x,-7.1%,-3.7%
ACC-2026,T-023,funded,"$75,000",5,100%,80%,0.4x,0.5x,2.9x,7.4x,-1.2%,-3.3%
ACC-2023,T-020,funded,"$75,000",8,75%,75%,2.2x,2.3x,2.3x,1.0x,-1.7%,-3.2%
ACC-2011,T-011,eval,"$25,000",5,60%,60%,3.4x,4.8x,3.4x,1.0x,-1.0%,-3.0%
ACC-2009,T-009,funded,"$25,000",5,60%,60%,3.4x,4.7x,11.3x,3.3x,-1.0%,-2.7%
ACC-2010,T-010,funded,"$25,000",5,60%,60%,4.7x,4.8x,3.4x,0.7x,-1.4%,-2.7%
ACC-2051,T-047,pro,"$25,000",5,80%,80%,1.2x,6.3x,6.4x,5.4x,-3.4%,-2.6%
ACC-2001,T-001,eval,"$100,000",10,100%,100%,28.1x,39.6x,2.8x,0.1x,-3.0%,-2.2%
ACC-2006,T-006,funded,"$150,000",10,100%,100%,2.4x,26.4x,2.6x,1.1x,-2.6%,-2.1%


In [17]:
# Who took more leverage today than ever before (above their historical max)
above_max = prop[prop["leverage_today"] > prop["leverage_max"] * 1.2]
above_max[cols_prop].sort_values("leverage_today_vs_normal", ascending=False).style.format(fmt)

,trader_id,type,balance,contract_limit,limit_usage_max,limit_usage_today,leverage_median,leverage_max,leverage_today,leverage_today_vs_normal,pnl_pct_worst,pnl_pct_today
account_id,,,,,,,,,,,,
ACC-2026,T-023,funded,"$75,000",5,100%,80%,0.4x,0.5x,2.9x,7.4x,-1.2%,-3.3%
ACC-2009,T-009,funded,"$25,000",5,60%,60%,3.4x,4.7x,11.3x,3.3x,-1.0%,-2.7%
ACC-2025,T-022,pro_plus,"$150,000",20,60%,160%,22.5x,22.6x,60.2x,2.7x,-0.7%,0.0%


**Reading:**

- **Loss as % of the account.** The largest losses relative to the account today are T-017 (−3.7%), T-023 (−3.3%), T-020 (−3.2%) and T-011 (−3.0%).
  T-017 has already had a −7.1% day, so it is not new for them. For T-023 and T-020 it is **more than double their previous worst day** (−1.2% and −1.7%).
  Prop firms' daily loss rules are often around 2–5% of the account; we do not have the real rules, but these traders are in that zone.
- **Leverage.** Traders risking far more than ever relative to their account today:
  - **T-022**: 60× the account, vs. 22× normally (previous max 22.6×). Also exceeded their contract limit.
  - **T-023**: 2.9× vs. 0.4× normally (7× more), from the MCL→CL switch.
  - **T-009**: 11.3× vs. 3.4× normally, from the MES→ES switch.
- **High leverage is not always unusual.** T-001, T-004 and T-007 routinely run 17–40× their account. For them 28× today is normal; for T-023, 3× is extraordinary. Again, what matters is the comparison with themselves.

### 8.3 Add these signals to the evidence

In [18]:
by_trader = prop.groupby("trader_id").agg(
    limit_usage_today=("limit_usage_today", "max"),
    limit_usage_max_hist=("limit_usage_max", "max"),
    leverage_today=("leverage_today", "max"),
    leverage_max_hist=("leverage_max", "max"),
    pnl_pct_today=("pnl_pct_today", "min"),
    pnl_pct_worst_hist=("pnl_pct_worst", "min"),
)
cmp = cmp.drop(columns=by_trader.columns, errors="ignore").join(by_trader)


def risk_evidence(r):
    out = []
    if r.limit_usage_today > 1:
        when = "for the first time" if r.limit_usage_max_hist <= 1 else "again (they do it often)"
        out.append(f"exceeded their contract limit {when} ({r.limit_usage_today:.0%} of the limit)")
    if r.leverage_today > 1.2 * r.leverage_max_hist:
        out.append(f"put {r.leverage_today:.1f}x their account at risk, more than ever (previous max {r.leverage_max_hist:.1f}x)")
    if r.pnl_pct_today < 1.5 * r.pnl_pct_worst_hist and r.pnl_pct_today < -0.02:
        out.append(f"lost {-r.pnl_pct_today:.1%} of the account (previous worst day {-r.pnl_pct_worst_hist:.1%})")
    return "; ".join(out)


cmp["risk_evidence"] = cmp.apply(risk_evidence, axis=1)
with pd.option_context("display.max_colwidth", 200):
    display(cmp.loc[cmp["risk_evidence"] != "", ["net_pnl", "pnl_z", "risk_evidence"]].sort_values("pnl_z"))

,net_pnl,pnl_z,risk_evidence
trader_id,,,
T-018,"-1,516.80",-8.91,lost 2.0% of the account (previous worst day 1.0%)
T-023,"-2,486.10",-5.45,"put 2.9x their account at risk, more than ever (previous max 0.5x); lost 3.3% of the account (previous worst day 1.2%)"
T-009,-669.95,-4.97,"put 11.3x their account at risk, more than ever (previous max 4.7x); lost 2.7% of the account (previous worst day 1.0%)"
T-011,-758.55,-4.62,lost 3.0% of the account (previous worst day 1.0%)
T-020,"-2,397.65",-4.24,lost 3.2% of the account (previous worst day 1.7%)
T-010,-663.85,-3.36,lost 2.7% of the account (previous worst day 1.4%)
T-013,-100.00,-1.85,exceeded their contract limit again (they do it often) (180% of the limit)
T-022,2.20,-0.35,"exceeded their contract limit for the first time (160% of the limit); put 60.2x their account at risk, more than ever (previous max 22.6x)"


## 9. Final score and top ten

### Why not add up every signal

Many signals describe **the same event**. T-023's switch from MCL to CL lights up five alarms at once (unusual loss, worst day, new product, more risk, % of account lost). If every alarm added a point, they would rank first for counting the same thing five times.

So the score answers **three independent questions**, 0 to 3 points each:

| Question | Measure | 0 | 1 | 2 | 3 |
| --- | --- | --- | --- | --- | --- |
| **A. Unusual loss for this trader?** | `pnl_z` (today's P&L vs. their mean, in their own standard deviations) | > −2 | −2 to −3 | −3 to −5 | ≤ −5 |
| **B. More risk than ever before?** | today's leverage / their historical max | < 1.2x | — | 1.2x to 2x | > 2x |
| **C. Broke a rule for the first time?** | exceeded `max_position_size` and never had before | no | — | — | yes |

**Score = A + B + C** (0 to 9). **Tie-break:** most negative `pnl_z` first (money already lost outweighs potential risk).

The full list of decision rules is in the README.

In [19]:
def loss_points(z, cuts=(-2, -3, -5)):
    if pd.isna(z):
        return 0
    return int(z <= cuts[0]) + int(z <= cuts[1]) + int(z <= cuts[2])


def risk_points(ratio, cuts=(1.2, 2.0)):
    if pd.isna(ratio) or ratio < cuts[0]:
        return 0
    return 3 if ratio > cuts[1] else 2


def rule_points(r):
    return 3 if (r.limit_usage_today > 1 and r.limit_usage_max_hist <= 1) else 0


def score(df, loss_cuts=(-2, -3, -5), risk_cuts=(1.2, 2.0)):
    out = df.copy()
    out["risk_vs_own_max"] = out["leverage_today"] / out["leverage_max_hist"]
    out["A_loss"] = out["pnl_z"].apply(loss_points, cuts=loss_cuts)
    out["B_risk"] = out["risk_vs_own_max"].apply(risk_points, cuts=risk_cuts)
    out["C_rule"] = out.apply(rule_points, axis=1)
    out["score"] = out["A_loss"] + out["B_risk"] + out["C_rule"]
    return out.sort_values(["score", "pnl_z"], ascending=[False, True])


scored = score(cmp)
top10 = scored.head(10).copy()
top10.insert(0, "rank", range(1, 11))
top10["full_evidence"] = [
    "; ".join(x for x in (e, er) if x) for e, er in zip(top10["evidence"], top10["risk_evidence"])
]
with pd.option_context("display.max_colwidth", 400):
    display(top10[["rank", "A_loss", "B_risk", "C_rule", "score", "net_pnl", "pnl_z",
                   "risk_vs_own_max", "full_evidence"]])

,rank,A_loss,B_risk,C_rule,score,net_pnl,pnl_z,risk_vs_own_max,full_evidence
trader_id,,,,,,,,,
T-023,1,3,3,0,6,"-2,486.10",-5.45,5.95,"lost $2,486; normally moves ±$387 (z=-5.4); already their worst session (2.7x their previous worst day); traded a product they had never used: CL; same contract count (0.9x) but 7.4x more money at risk than normal, more than ever before; holds an open position losing $2,340; put 2.9x their account at risk, more than ever (previous max 0.5x); lost 3.3% of the account (previous worst day 1.2%)"
T-022,2,0,3,3,6,2.20,-0.35,2.67,"2.7x more money at risk than normal, more than ever before; 10 orders vs. the 6 they usually place in a day; exceeded their contract limit for the first time (160% of the limit); put 60.2x their account at risk, more than ever (previous max 22.6x)"
T-009,3,2,3,0,5,-669.95,-4.97,2.38,"lost $670; normally moves ±$121 (z=-5.0); already their worst session (2.7x their previous worst day); traded a product they had never used: ES; same contract count (1.2x) but 3.3x more money at risk than normal, more than ever before; put 11.3x their account at risk, more than ever (previous max 4.7x); lost 2.7% of the account (previous worst day 1.0%)"
T-018,4,3,0,0,3,"-1,516.80",-8.91,0.67,"lost $1,517; normally moves ±$158 (z=-8.9); already their worst session (4.3x their previous worst day); lost 2.0% of the account (previous worst day 1.0%)"
T-019,5,3,0,0,3,"-1,559.20",-5.08,1.00,"lost $1,559; normally moves ±$305 (z=-5.1); already their worst session (3.2x their previous worst day)"
T-011,6,2,0,0,2,-758.55,-4.62,0.71,lost $759; normally moves ±$166 (z=-4.6); already their worst session (3.1x their previous worst day); lost 3.0% of the account (previous worst day 1.0%)
T-012,7,2,0,0,2,-724.10,-4.28,1.00,lost $724; normally moves ±$174 (z=-4.3); already their worst session (2.3x their previous worst day)
T-020,8,2,0,0,2,"-2,397.65",-4.24,1.00,"lost $2,398; normally moves ±$542 (z=-4.2); already their worst session (1.9x their previous worst day); 10 orders vs. the 6 they usually place in a day; lost 3.2% of the account (previous worst day 1.7%)"
T-010,9,2,0,0,2,-663.85,-3.36,0.71,lost $664; normally moves ±$180 (z=-3.4); already their worst session (1.8x their previous worst day); lost 2.7% of the account (previous worst day 1.4%)


In [20]:
# Just outside the list
scored.iloc[10:15][["A_loss", "B_risk", "C_rule", "score", "net_pnl", "pnl_z", "risk_vs_own_max"]]

,A_loss,B_risk,C_rule,score,net_pnl,pnl_z,risk_vs_own_max
trader_id,,,,,,,
T-013,0,0,0,0,-100.00,-1.85,1.00
T-029,0,0,0,0,-413.90,-1.58,0.75
T-007,0,0,0,0,"-3,067.40",-1.49,0.90
T-044,0,0,0,0,-559.92,-1.42,0.25
T-006,0,0,0,0,"-3,214.40",-1.29,0.10


### T-022: the case a loss-based list would miss

In [21]:
t22 = acct[acct.trader_id == "T-022"].groupby("session").agg(
    instrument=("instrument_symbol", lambda s: ",".join(sorted(set(s)))),
    max_contracts=("max_position", "max"), max_exposure_usd=("max_notional_usd", "max"), net_pnl=("net_pnl_usd", "sum"),
)
t22["limit"] = accounts.loc[accounts.trader_id == "T-022", "max_position_size"].iloc[0]
t22

,instrument,max_contracts,max_exposure_usd,net_pnl,limit
session,,,,,
2026-08-10,ES,12,"3,365,250.00",649.00,20
2026-08-11,ES,11,"3,086,875.00",-234.30,20
2026-08-12,ES,12,"3,369,000.00",365.80,20
2026-08-13,ES,12,"3,371,100.00",65.60,20
2026-08-14,ES,12,"3,372,000.00",753.20,20
2026-08-17,ES,12,"3,374,250.00",515.60,20
2026-08-18,ES,12,"3,376,200.00",-126.00,20
2026-08-19,ES,12,"3,377,700.00",224.10,20
2026-08-20,ES,12,"3,379,350.00",224.10,20


T-022 trades **ES** (the full-size S&P contract) and always used 12 contracts out of a limit of 20. Today they reached **32**: over their limit for the first time, with exposure going from ~$3.4 million to **$9 million** (60× their $150 thousand account). They made $2 today, so ranking by losses would put them at the bottom. Because they trade full-size contracts, the breach does not depend on how micros are counted.

### Outside the ranking, but worth a look

Not "unusually bad sessions", but hiding them would be worse:

In [22]:
outside = pd.DataFrame([
    {"trader_id": "T-013", "reason": "Exceeded their contract limit in 7 of 12 sessions (180% today). Recurring, not unusual: "
     "a controls failure, not a bad day. Trades MNQ (micro): if the firm counts 10 micros = 1 contract, it would not be a breach."},
    {"trader_id": "T-024", "reason": f"First day of trading; lost ${-excluded.loc['T-024', 'pnl_today']:,.0f}. No history to compare against."},
    {"trader_id": "T-025", "reason": "First day; the account shows a $0 balance (data to verify)."},
    {"trader_id": "T-026", "reason": "Only 1 session of history."},
])
with pd.option_context("display.max_colwidth", 300):
    display(outside)

,trader_id,reason
0,T-013,"Exceeded their contract limit in 7 of 12 sessions (180% today). Recurring, not unusual: a controls failure, not a bad day. Trades MNQ (micro): if the firm counts 10 micros = 1 contract, it would not be a breach."
1,T-024,"First day of trading; lost $1,580. No history to compare against."
2,T-025,First day; the account shows a $0 balance (data to verify).
3,T-026,Only 1 session of history.


## 10. Does the top ten depend on the thresholds?

Move the thresholds to other, equally reasonable values and count how many of the ten stay.

In [23]:
base_set = set(top10.index)
variants = {
    "base: loss (−2,−3,−5), risk (1.2x, 2x)": dict(),
    "stricter loss (−2.5,−3.5,−5.5)": dict(loss_cuts=(-2.5, -3.5, -5.5)),
    "looser loss (−1.5,−2.5,−4.5)": dict(loss_cuts=(-1.5, -2.5, -4.5)),
    "stricter risk (1.5x, 3x)": dict(risk_cuts=(1.5, 3.0)),
    "looser risk (1.1x, 1.5x)": dict(risk_cuts=(1.1, 1.5)),
}
rows = []
for name, kw in variants.items():
    t = list(score(cmp, **kw).head(10).index)
    rows.append({"variant": name, "in_common_with_base": len(base_set & set(t)),
                 "enter": ",".join(sorted(set(t) - base_set)), "leave": ",".join(sorted(base_set - set(t))),
                 "top3": ",".join(t[:3])})
pd.DataFrame(rows)

,variant,in_common_with_base,enter,leave,top3
0,"base: loss (−2,−3,−5), risk (1.2x, 2x)",10,,,"T-023,T-022,T-009"
1,"stricter loss (−2.5,−3.5,−5.5)",10,,,"T-022,T-023,T-009"
2,"looser loss (−1.5,−2.5,−4.5)",10,,,"T-023,T-009,T-022"
3,"stricter risk (1.5x, 3x)",10,,,"T-023,T-022,T-009"
4,"looser risk (1.1x, 1.5x)",10,,,"T-023,T-022,T-009"


The same ten traders appear in every variant, and the top three are always T-023, T-022 and T-009 (in slightly different order). The list depends on the three questions, not on the exact thresholds.

## 11. Does the ranking work? Backtest with recall@10

**Label (chosen in Task 2):** a trader got into trouble if they **broke a rule in any of the next 3 sessions**.

This dataset only has one real rule (`max_position_size`). It has no daily loss limit, so we use a **proxy**:

> **Broke a rule** = in that session they held more contracts than their limit **or** lost more than **2% of the account**.

**Metric: recall@10.** Of all the traders who broke a rule in the next 3 sessions, how many were on that day's list of ten? In other words: how many of the people heading for trouble did we see coming?

**How the backtest works:**
1. Stand on a past day *t* and build the ranking **using only data up to that day**, with the same rules as today (5-session minimum history, same score).
2. Look at the next 3 sessions and see who broke a rule.
3. Compute recall@10 and repeat for every possible day.

The days that can be evaluated need 5 sessions of history before and 3 after: **only 4 days** (17–20 August). The results are **indicative, not conclusive**.

A recall number alone says nothing, so we compare it against:
- **Simple ranking:** the 10 who lost the most dollars that day.
- **By % of account:** the 10 who lost the largest share of their account that day.
- **Random:** what 10 random picks would get (10 / number of eligible traders).

In [24]:
full = trader_sessions(acct).merge(size, on=["trader_id", "session"], how="left")
sessions = sorted(full["session"].unique())

# Label per account and session: did they break a rule that session? (proxy)
LOSS_THRESHOLD = -0.02
risk["broke_rule"] = (risk["limit_usage"] > 1) | (risk["pnl_pct_account"] <= LOSS_THRESHOLD)
broke = risk.groupby(["trader_id", "session"])["broke_rule"].any()


def score_day(t):
    """Ranking for day t using only data up to t (same rules as today's ranking)."""
    h = full[full["session"] < t]
    n_hist = h.groupby("trader_id")["session"].nunique()
    eligible = n_hist[n_hist >= 5].index
    day = full[(full["session"] == t) & full["trader_id"].isin(eligible)].set_index("trader_id")
    g = h[h["trader_id"].isin(eligible)].groupby("trader_id")

    r_h = risk[(risk["session"] < t)].groupby("account_id")
    r_t = risk[risk["session"] == t].set_index("account_id")
    acc = pd.DataFrame({
        "trader_id": r_t["trader_id"],
        "limit_usage_today": r_t["limit_usage"], "limit_usage_max_hist": r_h["limit_usage"].max(),
        "leverage_today": r_t["leverage"], "leverage_max_hist": r_h["leverage"].max(),
        "pnl_pct_today": r_t["pnl_pct_account"],
    }).groupby("trader_id").agg(
        limit_usage_today=("limit_usage_today", "max"), limit_usage_max_hist=("limit_usage_max_hist", "max"),
        leverage_today=("leverage_today", "max"), leverage_max_hist=("leverage_max_hist", "max"),
        pnl_pct_today=("pnl_pct_today", "min"),
    )
    out = day[["net_pnl_usd"]].rename(columns={"net_pnl_usd": "net_pnl"}).join(acc)
    out["pnl_z"] = (out["net_pnl"] - g["net_pnl_usd"].mean()) / g["net_pnl_usd"].std()
    return score(out)


def recall_at_10(top, positives):
    return len(set(top) & positives) / len(positives) if positives else np.nan


rows = []
for i, t in enumerate(sessions):
    future = sessions[i + 1:i + 4]
    if len(future) < 3:
        continue
    s = score_day(t)
    if len(s) == 0:
        continue
    positives = {
        tr for tr in s.index
        if broke.reindex([(tr, f) for f in future]).fillna(False).any()
    }
    rows.append({
        "day": t.date(),
        "eligible": len(s),
        "broke_rule_next_3d": len(positives),
        "our_score": recall_at_10(s.head(10).index, positives),
        "most_dollars_lost": recall_at_10(s.nsmallest(10, "net_pnl").index, positives),
        "most_pct_account_lost": recall_at_10(s.nsmallest(10, "pnl_pct_today").index, positives),
        "random": 10 / len(s),
    })
bt = pd.DataFrame(rows)
bt

,day,eligible,broke_rule_next_3d,our_score,most_dollars_lost,most_pct_account_lost,random
0,2026-08-17,28,4,0.00,0.00,0.00,0.36
1,2026-08-18,35,5,0.40,0.40,0.40,0.29
2,2026-08-19,44,4,0.00,0.00,0.00,0.23
3,2026-08-20,45,14,0.21,0.36,0.36,0.22


In [25]:
summary = bt[["our_score", "most_dollars_lost", "most_pct_account_lost", "random"]].mean().rename("mean recall@10")
summary.to_frame().style.format("{:.0%}")

,mean recall@10
our_score,15%
most_dollars_lost,19%
most_pct_account_lost,19%
random,27%


### Who did we miss?

Low recall can mean the ranking is bad **or** that the label counts things the ranking deliberately ignores. Who broke a rule without being on the list?

In [26]:
missed = []
for i, t in enumerate(sessions):
    future = sessions[i + 1:i + 4]
    if len(future) < 3:
        continue
    s = score_day(t)
    if len(s) == 0:
        continue
    top = set(s.head(10).index)
    for tr in s.index:
        broken_days = [f for f in future if broke.get((tr, f), False)]
        if broken_days and tr not in top:
            missed.append({"day": t.date(), "trader_id": tr, "rank": list(s.index).index(tr) + 1,
                           "broke_on": ",".join(str(f.date()) for f in broken_days)})
missed = pd.DataFrame(missed)
missed.groupby("trader_id").agg(times_missed=("day", "size"), mean_rank=("rank", "mean")).sort_values("times_missed", ascending=False)

,times_missed,mean_rank
trader_id,,
T-013,4,27.00
T-006,3,25.00
T-001,2,24.00
T-002,2,31.00
T-004,2,28.50
T-009,1,25.00
T-010,1,36.00
T-011,1,15.00
T-017,1,31.00


In [27]:
# Why did they break the rule? (contract limit vs. loss > 2%)
reason = risk[risk["broke_rule"]].assign(
    by_contracts=lambda x: x["limit_usage"] > 1, by_loss=lambda x: x["pnl_pct_account"] <= LOSS_THRESHOLD,
).groupby("trader_id")[["by_contracts", "by_loss"]].sum()
reason.loc[reason.index.isin(missed["trader_id"])].sort_values("by_contracts", ascending=False)

,by_contracts,by_loss
trader_id,,
T-013,7,0
T-022,1,0
T-001,0,4
T-002,0,2
T-004,0,5
T-006,0,3
T-009,0,1
T-010,0,1
T-011,0,1


### Sensitivity to the loss threshold

The 2% is our choice. Does the conclusion change at 1.5% or 3%?

In [28]:
def backtest(threshold):
    r = (risk["limit_usage"] > 1) | (risk["pnl_pct_account"] <= threshold)
    rp = r.groupby([risk["trader_id"], risk["session"]]).any()
    res = []
    for i, t in enumerate(sessions):
        future = sessions[i + 1:i + 4]
        if len(future) < 3:
            continue
        s = score_day(t)
        if len(s) == 0:
            continue
        pos = {tr for tr in s.index if rp.reindex([(tr, f) for f in future]).fillna(False).any()}
        res.append({
            "our_score": recall_at_10(s.head(10).index, pos),
            "most_dollars_lost": recall_at_10(s.nsmallest(10, "net_pnl").index, pos),
            "most_pct_account_lost": recall_at_10(s.nsmallest(10, "pnl_pct_today").index, pos),
            "random": 10 / len(s),
            "positives": len(pos),
        })
    return pd.DataFrame(res).mean()


sens = pd.DataFrame({f"loss > {abs(u):.1%}": backtest(u) for u in (-0.015, -0.02, -0.03)}).T
sens.style.format({c: "{:.0%}" for c in sens.columns if c != "positives"} | {"positives": "{:.1f}"})

,our_score,most_dollars_lost,most_pct_account_lost,random,positives
loss > 1.5%,21%,24%,25%,27%,9.0
loss > 2.0%,15%,19%,19%,27%,6.8
loss > 3.0%,20%,24%,24%,27%,3.5


### An honest reading of the backtest

**No ranking beats random**, ours included (15%) and the simple ones (19%), against ~27% for 10 random picks. The conclusion is the same at 1.5%, 2% or 3%.

Before concluding that "the ranking doesn't work", three things:

1. **There is very little evidence.** 4 days and ~27 positive cases in total. One day with 14 positives outweighs the other three combined. This cannot tell "worse than random" apart from noise.
2. **Most of the misses are traders we chose not to look at.**
   - **T-013** breaks their contract limit 7 times: excluded on purpose because it is recurring.
   - **T-001, T-004, T-006, T-002, T-017** often lose more than 2% of their account. **For them that is normal**, so our ranking (relative to each trader) does not flag them. But a fixed loss rule catches them anyway.
3. **In this data, one bad day does not predict another.** The simple rankings ("who lost the most today") do not beat random either; the "planted" problems seem concentrated in the last session, which cannot be evaluated because there are no later sessions.

**The important lesson is a tension between two questions:**

- *"Who is having an unusual day for them?"* (what Task 1 asks) → a relative ranking.
- *"Who is about to break a rule?"* (what costs the firm money) → depends on **how close they are to the limit in absolute terms**, even if that is normal for them.

A trader who loses 3% of the account every day is "normal" in the first sense and the most likely to breach in the second. **A ranking the firm can act on probably needs both**: what is unusual for each trader *and* the distance to their real limits. That would be the next step, with each account's actual rules.